In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import joblib
import os


print('Contents of /content/drive/MyDrive/F1_2026_Project/models/')
print(os.listdir('/content/drive/MyDrive/F1_2026_Project/models/'))

# List contents of the 'models/encoders' directory
print('\nContents of /content/drive/MyDrive/F1_2026_Project/models/encoders/')
print(os.listdir('/content/drive/MyDrive/F1_2026_Project/models/encoders/'))


scaler = joblib.load('/content/drive/MyDrive/F1_2026_Project/models/scaler.joblib')

Contents of /content/drive/MyDrive/F1_2026_Project/models/
['le_track.joblib', 'le_engine.joblib', 'encoders', 'f1_v1.pth', 'scaler.joblib']

Contents of /content/drive/MyDrive/F1_2026_Project/models/encoders/
['le_track.joblib', 'scaler.joblib', 'le_engine.joblib', 'y_scaler.joblib']


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from google.colab import drive

drive.mount('/content/drive')

class F1Oracle(nn.Module):
  def __init__(self, input_dim):
    super(F1Oracle, self).__init__()
    self.layers = nn.Sequential(
        nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
  def forward(self, x): return self.layers(x)

model = F1Oracle(input_dim = 3)
model.load_state_dict(torch.load('/content/drive/MyDrive/F1_2026_Project/models/f1_v1.pth'))
model.eval()
print("Oracle is ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Oracle is ready


In [ ]:
import joblib

# 1. Load the "saved" encoders and scaler
encoder_path = '/content/drive/MyDrive/F1_2026_Project/models/'
le_track = joblib.load(f'{encoder_path}le_track.joblib')
le_engine = joblib.load(f'{encoder_path}le_engine.joblib')
scaler = joblib.load(f'{encoder_path}scaler.joblib')

In [ ]:
import joblib
import numpy as np


le_track = joblib.load('/content/drive/MyDrive/F1_2026_Project/models/encoders/le_track.joblib')
le_engine = joblib.load('/content/drive/MyDrive/F1_2026_Project/models/encoders/le_engine.joblib')
scaler = joblib.load('/content/drive/MyDrive/F1_2026_Project/models/encoders/scaler.joblib')


new_engines = ['Audi', 'RB-Ford']
for eng in new_engines:
    if eng not in le_engine.classes_:
        le_engine.classes_ = np.append(le_engine.classes_, eng)


if 'Australian Grand Prix' not in le_track.classes_:
     le_track.classes_ = np.append(le_track.classes_, 'Australian Grand Prix')

In [ ]:
is_corrupted = torch.stack([torch.isnan(p).any() for p in model.parameters()]).any()
print(f"Is the model corrupted with NaNs? {is_corrupted.item()}")

Is the model corrupted with NaNs? False


In [ ]:
import fastf1
import pandas as pd
import numpy as np
import torch

# 1. Fetch the Live Qualifying session (Change the race name to whatever is next!)
print("Fetching Live Qualifying Data...")
session = fastf1.get_session(2026, 'Chinese Grand Prix', 'Q')
session.load(laps=False, telemetry=False) # False makes it load instantly since we only need the final grid
results = session.results


grid_data = []
for _, driver in results.iterrows():
    # If a driver didn't set a time (like a Q1 crash), default them to P20
    grid_pos = driver['Position'] if not pd.isna(driver['Position']) else 20.0

    grid_data.append({
        'Driver': driver['Abbreviation'],
        'GridPosition': int(grid_pos),
        'Team': driver['TeamName']
    })

scenario_2026 = pd.DataFrame(grid_data)

#Add the Engine and Track mapping dynamically
def get_engine(team_name):
    name = str(team_name).lower()
    if 'audi' in name or 'sauber' in name or 'stake' in name: return 'Audi'
    if 'bull' in name or 'rb' in name: return 'RB-Ford'
    if 'aston' in name: return 'Honda'
    if 'ferrari' in name or 'haas' in name or 'cadillac' in name: return 'Ferrari'
    return 'Mercedes' # Catch-all for Mercedes, McLaren, Williams, Alpine

scenario_2026['Engine'] = scenario_2026['Team'].apply(get_engine)
scenario_2026['Track'] = 'Chinese Grand Prix'

# Sneak the Chinese GP track into the encoder (since your V1 scraper skipped Sprint weekends!)
if 'Chinese Grand Prix' not in le_track.classes_:
    le_track.classes_ = np.append(le_track.classes_, 'Chinese Grand Prix')


scenario_2026['Track_Enc'] = le_track.transform(scenario_2026['Track'])
scenario_2026['Engine_Enc'] = le_engine.transform(scenario_2026['Engine'])


X_live = scenario_2026[['GridPosition', 'Track_Enc', 'Engine_Enc']].values.astype(np.float32)
X_live_scaled = scaler.transform(X_live)
X_tensor_live = torch.from_numpy(X_live_scaled)

print("✅ Live Grid Data Prepped! Ready for predictions.")

core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for weather_data
INFO:fastf1.fastf1.req:Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
INFO:fastf1.fastf1.req:Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '81', '1', '10', '3', '6', '87', '27', '43', '31', '30', '41', '5', '55', '23', '14', '77', '18', '11']
INFO:fastf1.fastf1.core:Finished loading data for 22 drivers: ['12', '63', '44', '16', '81', '1', '10', '3', '6', '87', '27', '43', '31', '30', '41', '5', '55', '23', '14', '77', 

Fetching Live Qualifying Data...
✅ Live Grid Data Prepped! Ready for predictions.


In [ ]:
# Make sure you run the cell defining the model first!
model.eval()

with torch.no_grad():

    predictions = model(X_tensor_live).numpy()


scenario_2026['PredictedFinish'] = predictions.flatten()


scenario_2026['PredictedFinish'] = scenario_2026['PredictedFinish'].clip(1, 20).round(2)


results = scenario_2026.sort_values(by='PredictedFinish')

print("--- GP PREDICTED RESULTS ---")
print(results[['Driver', 'GridPosition', 'Engine', 'PredictedFinish']])

--- GP PREDICTED RESULTS ---
   Driver  GridPosition    Engine  PredictedFinish
0     ANT             1  Mercedes             2.71
1     RUS             2  Mercedes             3.41
2     HAM             3   Ferrari             4.50
3     LEC             4   Ferrari             5.38
4     PIA             5  Mercedes             6.55
5     NOR             6  Mercedes             7.67
6     GAS             7  Mercedes             8.58
9     BEA            10   Ferrari             8.79
7     VER             8   RB-Ford             9.41
8     HAD             9   RB-Ford             9.95
12    OCO            13   Ferrari            10.50
11    COL            12  Mercedes            12.80
13    LAW            14   RB-Ford            12.85
10    HUL            11      Audi            12.99
19    BOT            20   Ferrari            13.25
14    LIN            15   RB-Ford            13.36
21    PER            22   Ferrari            14.01
18    ALO            19     Honda            14.03
16